# Atom-Versatz im zeitunabhaengigen Profil

Das **Atom** wandert aus der Mitte nach aussen, das Lichtfeld steht still.
Mit ihm wandern alle Regionen, ueber die ausgewertet wird. Gerechnet wird mit
`lib/atom_offset.py` - hier steht nur, was eingetragen, gezeichnet und
gespeichert wird.

Ergebnis: ein PDF und eine CSV in `Notebook_Ergebnisse/`.

In [1]:
# ====================  PARAMETER  ====================================
WAIST_UM     = 1.04      # Waist in der Atomebene (um)
WIDTH_MHZ    = 0.37      # Tonabstand; die Spannweite d des 3x4-Arrays folgt daraus
R_X, R_Y     = 0.97, 1.16   # Amplitudenverhaeltnisse
PROFIL       = "airy"    # "airy" oder "gauss"

ATOM_T_UK    = 17.0      # Atomtemperatur (uK)  -> sigma_atom
NU_R_KHZ     = 60.4      # radiale Fallenfrequenz (kHz)

RICHTUNG     = "x"       # "x" horizontal, "y" vertikal, "xy" diagonal
R_MAX_ANTEIL = 0.5       # 0.5 = bis an den aeusseren Ton (Rand des Tonarrays)
N_PUNKTE     = 41        # Stuetzstellen von r = 0 bis r_max
HART         = False     # True: harte und kombinierte Groessen zusaetzlich (doppelte Rechenzeit)
LOG_Y        = False     # log-Achse; mit HART = True noetig, sonst verschwindet die gewichtete Kurve

NAME         = "AtomOffset_horizontal"   # Dateiname-Kern
# =====================================================================

import sys
from pathlib import Path
BASE = Path.cwd()                 # Ordner Combinated_Optimization
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
OUT = BASE / "Notebook_Ergebnisse"
OUT.mkdir(exist_ok=True)
print("Ausgabe nach:", OUT)

Ausgabe nach: /mnt/user-data/uploads/Multitone_Simulation/Amplitudes/Combinated_Optimization/Notebook_Ergebnisse


In [2]:
# ====================  RECHNEN  ======================================
import numpy as np
from lib import atom_offset as ao

params = dict(ao.DEFAULTS,
              waist=WAIST_UM * 1e-6, width=WIDTH_MHZ * 1e6,
              r_x=R_X, r_y=R_Y, profile=PROFIL,
              atom_temperature=ATOM_T_UK * 1e-6, trap_freq_r=NU_R_KHZ * 1e3,
              hart=HART)

opt   = ao.make_optimizer(params)
d_um  = ao.width_um(params, opt)          # Spannweite des Tonarrays
r_max = R_MAX_ANTEIL * d_um * 1e-6
print(f"Spannweite Tonarray d = {d_um:.4f} um   ->   r_max = {r_max*1e6:.4f} um")
print(f"sigma_atom = {opt.sigma_atom*1e9:.1f} nm,   pitch = {opt.pitch*1e6:.3f} um")

def zeige(i, g):
    if i % 10 == 0 or i == g:
        print(f"  {i}/{g}")
    return True

res = ao.sweep(params, r_max, n=N_PUNKTE, richtungen=(RICHTUNG,), progress=zeige)
print(f"fertig in {res['dauer_s']:.1f} s")

Spannweite Tonarray d = 2.3370 um   ->   r_max = 1.1685 um
sigma_atom = 107.6 nm,   pitch = 5.288 um


  10/41


  20/41


  30/41


  40/41


  41/41
fertig in 13.4 s


In [3]:
# ====================  PLOT  =========================================
import matplotlib.pyplot as plt

FIG_W_IN, PT = 16.0 / 2.54, 10.0          # 16 cm = \textwidth, 10 pt Fliesstext
FAMILIEN = [("hart",     "hart",        "#332288"),
            ("weighted", "gewichtet",   "#D55E00"),
            ("kombi",    "kombiniert",  "#009E73")]
if not HART:
    FAMILIEN = [f for f in FAMILIEN if f[0] == "weighted"]

r_um = res["offset_um"]
mehrere = len(FAMILIEN) > 1
fig, (ax_u, ax_c) = plt.subplots(
    1, 2, figsize=(FIG_W_IN, (0.46 if mehrere else 0.42) * FIG_W_IN))
for ax, groesse, sym in ((ax_u, "uniformity", "U"), (ax_c, "crosstalk", r"\eta")):
    for fam, name, farbe in FAMILIEN:
        y = res[f"{groesse}_{fam}__{RICHTUNG}"] * 100.0
        ax.plot(r_um, y, color=farbe, lw=1.4, label=name)
    ax.set_xlabel(r"$r$ ($\mathrm{\mu}$m)")
    ax.set_ylabel(rf"${sym}$ (%)")
    ax.set_xlim(r_um[0], r_um[-1])
    if LOG_Y:
        ax.set_yscale("log")
    ax.grid(alpha=0.25, lw=0.5)
    # zweite Achse: Versatz in Einheiten der halben Spannweite (1 = aeusserer Ton)
    sek = ax.secondary_xaxis("top", functions=(lambda v: v / (0.5 * d_um),
                                               lambda v: v * (0.5 * d_um)))
    sek.set_xlabel(r"$r\,/\,(d/2)$")
# Legende nur, wenn es etwas zu unterscheiden gibt - dann unter beiden
# Plots, so verdeckt sie keine Kurve.
if mehrere:
    fig.legend(*ax_u.get_legend_handles_labels(), loc="lower center",
               ncol=len(FAMILIEN), frameon=False)

for o in fig.findobj(plt.Text):
    o.set_fontsize(PT)
for ax in fig.axes:
    ax.tick_params(labelsize=PT)
fig.tight_layout(rect=(0, 0.10 if mehrere else 0.0, 1, 1))

pdf = OUT / f"{NAME}_w{WAIST_UM:.2f}um_width{WIDTH_MHZ:.2f}MHz.pdf"
fig.savefig(pdf)
print("gespeichert:", pdf.name)

gespeichert: AtomOffset_horizontal_w1.04um_width0.37MHz.pdf


In [4]:
# ====================  DATEN SPEICHERN  ==============================
spalten = {"r_um": r_um, "r_pro_halbe_spannweite": r_um / (0.5 * d_um)}
for fam, _name, _farbe in FAMILIEN:
    for groesse in ("uniformity", "crosstalk"):
        spalten[f"{groesse}_{fam}_prozent"] = res[f"{groesse}_{fam}__{RICHTUNG}"] * 100.0
if HART:
    spalten["J_prozent"] = res[f"combined_score__{RICHTUNG}"] * 100.0

csv = OUT / f"{NAME}_w{WAIST_UM:.2f}um_width{WIDTH_MHZ:.2f}MHz.csv"
kopf = (f"# 3x4, waist {WAIST_UM} um, width {WIDTH_MHZ} MHz, r_x {R_X}, r_y {R_Y}, "
        f"{PROFIL}\n# Spannweite Tonarray d = {d_um:.4f} um, "
        f"sigma_atom = {opt.sigma_atom*1e9:.1f} nm, Richtung {RICHTUNG}\n")
with open(csv, "w", encoding="utf-8") as f:
    f.write(kopf + ",".join(spalten) + "\n")
    np.savetxt(f, np.column_stack(list(spalten.values())), delimiter=",", fmt="%.6g")
print("gespeichert:", csv.name)

# kurze Uebersicht: Anfang, Ende, Faktor
for name, werte in spalten.items():
    if name.endswith("prozent"):
        print(f"{name:34s} {werte[0]:8.4f}  ->  {werte[-1]:8.4f}   "
              f"(x{werte[-1]/werte[0]:.2f})")

gespeichert: AtomOffset_horizontal_w1.04um_width0.37MHz.csv
uniformity_weighted_prozent          0.4292  ->    3.7504   (x8.74)
crosstalk_weighted_prozent           1.3981  ->    2.0283   (x1.45)


## Notizen